In [1]:
import os
os.environ['HF_HOME'] = "/ssd_scratch/yuvraj.bodana/huggingface"
os.environ['HF_DATASETS_CACHE'] = "/ssd_scratch/yuvraj.bodana/huggingface"

In [2]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
from cka_metrics import cka, gram_linear, feature_space_linear_cka

In [3]:
np.random.seed(1337)
X = np.random.randn(100, 10)
Y = np.random.randn(100, 10) + X

In [4]:
cka_from_examples = cka(gram_linear(X), gram_linear(Y))
cka_from_features = feature_space_linear_cka(X, Y)

print('Linear CKA from Examples: {:.5f}'.format(cka_from_examples))
print('Linear CKA from Features: {:.5f}'.format(cka_from_features))
np.testing.assert_almost_equal(cka_from_examples, cka_from_features)

The values are: 316.51928904624316 and 659.6195229836956
Linear CKA from Examples: 0.55761
Linear CKA from Features: 0.55761


In [57]:
np.random.seed(1337)

x = np.random.randn(1,10)
y = np.random.randn(1,10)

np.array([*x,*y]).shape

(2, 10)

In [58]:
feature_space_linear_cka(*x,*y)

np.float64(0.23579444959743015)

In [56]:
x

array([[-0.70318731, -0.49028236, -0.32181433, -1.75507872,  0.20666447,
        -2.01126457, -0.55725071,  0.33721701,  1.54883597, -1.37073656]])

In [3]:
sentences_en = [
    "How are you?",
    "I am fine.",
    "What is your name?",
    "Good morning!",
    "See you later."
]
sentences_fr = [
    "आप कैसे हैं?",
"मैं ठीक हूँ।",
"आपका नाम क्या है?",
"सुप्रभात!",
"बाद में मिलते हैं।"
]

In [4]:
model = SentenceTransformer('l3cube-pune/indic-sentence-similarity-sbert')

In [5]:
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
)

In [6]:
embeddings_en = model.encode(sentences_en)
embeddings_fr = model.encode(sentences_fr)


In [7]:
embeddings_en[0].shape, embeddings_fr[0].shape

((768,), (768,))

In [15]:
embeddings_en[0]

array([-1.75672378e-02,  2.20376183e-03,  5.00781387e-02,  5.57885272e-03,
       -1.64918937e-02,  7.30773958e-04,  1.77332666e-02, -1.60317589e-02,
       -2.74200831e-02,  3.12398444e-03,  1.27677841e-03,  1.29553648e-02,
       -1.45942308e-02, -5.97487343e-03,  1.09818885e-02, -1.96975227e-02,
        3.55367386e-03,  1.70716085e-02,  4.99406550e-03,  1.62989721e-02,
        1.31198240e-03,  2.66886596e-02, -2.85561923e-02, -3.92795028e-03,
        2.97956839e-02, -6.79809740e-03, -1.61698333e-03,  3.47687234e-03,
        1.05397368e-03,  1.47294858e-02,  3.32399126e-04,  1.41537131e-03,
        1.14137465e-02,  4.75301407e-02,  1.70519259e-02, -1.99643392e-02,
        7.71687692e-03,  9.55466926e-03,  4.78884409e-04,  6.46826637e-04,
       -7.93774799e-03, -2.19909027e-02,  6.90475106e-03, -1.00184157e-02,
       -9.12845321e-03, -1.59246642e-02, -1.98773574e-03,  7.59068644e-03,
       -1.14159957e-02,  1.72321778e-03, -1.75408181e-02, -2.18868442e-03,
        1.63932350e-02,  

In [8]:
cos_similarities = [
    cosine_similarity([emb_en], [emb_fr])[0][0]
    for emb_en, emb_fr in zip(embeddings_en, embeddings_fr)
]
avg_cosine_similarity = sum(cos_similarities) / len(cos_similarities)

print(f"Average Cosine Similarity: {avg_cosine_similarity:.4f}")

Average Cosine Similarity: 0.8345


In [9]:
cka_score = cka(gram_linear(embeddings_en), gram_linear(embeddings_fr))

The values are: 0.3941061794757843 and 0.4407288134098053


In [10]:
cka_score

0.9849499

In [11]:
feature_cka = feature_space_linear_cka(embeddings_en, embeddings_fr)

In [12]:
feature_cka

0.984950138725567

In [16]:
gram_linear(embeddings_en)

array([[0.22300297, 0.02825677, 0.09636155, 0.05159992, 0.05185562],
       [0.02825677, 0.20875496, 0.02722936, 0.07294673, 0.02687077],
       [0.09636155, 0.02722936, 0.2306462 , 0.03963112, 0.06812026],
       [0.05159992, 0.07294673, 0.03963112, 0.2662898 , 0.05841863],
       [0.05185562, 0.02687077, 0.06812026, 0.05841863, 0.2874552 ]],
      dtype=float32)

In [19]:
cka_score = cka(gram_linear(np.array([embeddings_en[0]])), gram_linear(np.array([embeddings_fr[0]])))

The values are: 0.0 and 0.0


/home2/yuvrajsinh.bodana/Pruning/EXP/cka_metrics.py:94: RuntimeWarning: invalid value encountered in scalar divide
  return scaled_hsic / (normalization_x * normalization_y)


In [2]:
from datasets import load_dataset

In [3]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [4]:
ds['train']

Dataset({
    features: ['text'],
    num_rows: 36718
})

In [7]:
ds['train'][1:100]

{'text': [' = Valkyria Chronicles III = \n',
  '',
  ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n',
  " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game

In [18]:
counts = []
def count_tokens(example):
    # print(example,type(example))
    if example['text']:
        counts.append(example['text'].split())
        return example['text'].split()
    return 0

In [21]:
total=0
for x in ds['train']:
    total+=len(x['text'].split())
# ds["train"][0]['text']
print(total)

2051910


In [1]:
from torch.cuda import empty_cache